# Part 3 Travel Timing Recommendation System

This notebook develops a practical recommendation system that identifies suitable travel windows using historical traffic patterns from the cleaned and feature-engineered dataset produced in Part 2.

Because the dataset represents traffic observations from a single road corridor, the system focuses on recommending when to travel rather than selecting between alternative routes.

The recommendation design is informed by the modelling results from the previous notebooks. The Random Forest achieved stronger predictive performance than the neural network, while the SHAP analysis showed that hour and day-related variables were the main contributors to predicted traffic volume. The system therefore prioritises travel timing and day type, with weather used as an additional condition.

Recommendations are calculated from observed hourly traffic volumes for different day types and weather conditions rather than from direct Random Forest predictions. This keeps the recommendation logic transparent and prevents historical evidence from being presented as a real-time traffic forecast.

The system provides plain-language travel guidance supported by historical data. Its recommendations are intended to support journey planning and do not guarantee future traffic conditions.

In [1]:
import sys
import pandas as pd

print("Executable:", sys.executable)
print("Python:", sys.version)
print("pandas:", pd.__version__)

Executable: /usr/local/bin/python3
Python: 3.13.7 (v3.13.7:bcee1c32211, Aug 14 2025, 19:10:51) [Clang 16.0.0 (clang-1600.0.26.6)]
pandas: 3.0.5


In [2]:
from pathlib import Path
import logging
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent

if not (PROJECT_ROOT / "models").exists():
    PROJECT_ROOT = Path.cwd()

DATA_PATH = (
    PROJECT_ROOT.parent
    / "part2_python"
    / "data"
    / "processed"
    / "traffic_features.csv"
)

RECOMMENDATION_DIR = (
    PROJECT_ROOT / "recommendation_system"
)

LOG_PATH = PROJECT_ROOT / "part3.log"

RECOMMENDATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LOGGER = logging.getLogger(__name__)

if not LOGGER.handlers:
    LOGGER.setLevel(logging.INFO)

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | "
        "%(name)s | %(message)s"
    )

    console_handler = logging.StreamHandler(
        sys.stdout
    )
    console_handler.setFormatter(formatter)

    file_handler = logging.FileHandler(
        LOG_PATH,
        mode="a",
        encoding="utf-8",
    )
    file_handler.setFormatter(formatter)

    LOGGER.addHandler(console_handler)
    LOGGER.addHandler(file_handler)

LOGGER.info(
    "Travel-timing recommendation notebook started"
)

print("Dataset:", DATA_PATH)
print(
    "Recommendation directory:",
    RECOMMENDATION_DIR,
)

2026-09-25 04:13:57,924 | INFO | __main__ | Travel-timing recommendation notebook started
Dataset: /Users/misatoniwa/Downloads/smart-city-traffic-capstone/part2_python/data/processed/traffic_features.csv
Recommendation directory: /Users/misatoniwa/Downloads/smart-city-traffic-capstone/part3_machine_learning/recommendation_system


## Loading and preparing the historical traffic data

The recommendation system uses the cleaned and feature-engineered dataset produced in Part 2. Weather categories are reconstructed from the one-hot encoded weather columns, while observations are classified as either weekdays or weekends.

These variables allow hourly traffic patterns to be compared under different day and weather conditions.

In [3]:
try:
    data = pd.read_csv(DATA_PATH)
except FileNotFoundError as exc:
    LOGGER.error(
        "Feature dataset was not found at %s",
        DATA_PATH,
    )
    raise FileNotFoundError(
        "Run the Part 2 pipeline and feature-engineering "
        "script before running this notebook."
    ) from exc
except pd.errors.ParserError as exc:
    LOGGER.error(
        "Feature dataset could not be parsed: %s",
        exc,
    )
    raise

required_columns = {
    "hour",
    "is_weekend",
    "traffic_volume",
}

missing_columns = required_columns.difference(
    data.columns
)

if missing_columns:
    raise ValueError(
        "The dataset is missing required columns: "
        f"{sorted(missing_columns)}"
    )

weather_columns = [
    column
    for column in data.columns
    if column.startswith("weather_main_")
]

if not weather_columns:
    raise ValueError(
        "No one-hot encoded weather columns were found."
    )

data["Weather"] = (
    data[weather_columns]
    .idxmax(axis=1)
    .str.replace(
        "weather_main_",
        "",
        regex=False,
    )
)

data["Day_Type"] = data["is_weekend"].map(
    {
        0: "Weekday",
        1: "Weekend",
    }
)

LOGGER.info(
    "Loaded %d traffic observations with %d weather categories",
    len(data),
    data["Weather"].nunique(),
)

print("Dataset shape:", data.shape)
print(
    "Day types:",
    sorted(data["Day_Type"].dropna().unique()),
)
print(
    "Weather categories:",
    sorted(data["Weather"].dropna().unique()),
)

data[
    [
        "hour",
        "Day_Type",
        "Weather",
        "traffic_volume",
    ]
].head()

2026-09-25 04:14:33,660 | INFO | __main__ | Loaded 48187 traffic observations with 11 weather categories
Dataset shape: (48187, 36)
Day types: ['Weekday', 'Weekend']
Weather categories: ['Clear', 'Clouds', 'Drizzle', 'Fog', 'Haze', 'Mist', 'Rain', 'Smoke', 'Snow', 'Squall', 'Thunderstorm']


,hour,Day_Type,Weather,traffic_volume
0,9,Weekday,Clouds,5545
1,10,Weekday,Clouds,4516
2,11,Weekday,Clouds,4767
3,12,Weekday,Clouds,5026
4,13,Weekday,Clouds,4918


## Historical hourly traffic patterns

Average and median traffic volumes are calculated for each combination of day type, weather condition and hour. The number of historical observations is also retained so that recommendations are not based on very small samples.

A minimum of 30 observations is required before a weather-specific hourly pattern is used. If insufficient evidence is available for a selected weather condition, the system falls back to the broader hourly pattern for the selected day type.

In [4]:
MIN_OBSERVATIONS = 30

hourly_weather_patterns = (
    data.groupby(
        [
            "Day_Type",
            "Weather",
            "hour",
        ]
    )["traffic_volume"]
    .agg(
        Average_Traffic="mean",
        Median_Traffic="median",
        Observations="count",
    )
    .reset_index()
)

day_hour_patterns = (
    data.groupby(
        [
            "Day_Type",
            "hour",
        ]
    )["traffic_volume"]
    .agg(
        Average_Traffic="mean",
        Median_Traffic="median",
        Observations="count",
    )
    .reset_index()
)

reliable_weather_patterns = (
    hourly_weather_patterns[
        hourly_weather_patterns[
            "Observations"
        ] >= MIN_OBSERVATIONS
    ]
    .copy()
)

LOGGER.info(
    "Created %d weather-specific hourly patterns, "
    "of which %d met the minimum sample requirement",
    len(hourly_weather_patterns),
    len(reliable_weather_patterns),
)

print(
    "Minimum observations required:",
    MIN_OBSERVATIONS,
)
print(
    "Reliable weather-specific patterns:",
    len(reliable_weather_patterns),
)

reliable_weather_patterns.sort_values(
    "Average_Traffic"
).head(10).round(1)

2026-09-25 04:21:07,925 | INFO | __main__ | Created 455 weather-specific hourly patterns, of which 296 met the minimum sample requirement
Minimum observations required: 30
Reliable weather-specific patterns: 296


,Day_Type,Weather,hour,Average_Traffic,Median_Traffic,Observations
50,Weekday,Drizzle,2,292.4,280.0,59
122,Weekday,Mist,2,294.4,281.0,222
146,Weekday,Rain,2,299.3,286.0,179
26,Weekday,Clouds,2,300.1,286.0,341
178,Weekday,Snow,2,306.0,275.0,91
2,Weekday,Clear,2,307.0,295.0,457
74,Weekday,Fog,2,307.9,294.5,38
204,Weekday,Thunderstorm,2,308.7,300.0,35
98,Weekday,Haze,2,309.7,307.0,33
409,Weekend,Snow,4,339.3,339.0,38


## Travel-window recommendation function

The function below recommends lower-traffic travel windows for a selected day type and weather condition. Users can also specify the range of hours during which they are willing to travel.

Weather-specific patterns are used when enough historical observations are available. Otherwise, the function falls back to the broader weekday or weekend pattern. This prevents recommendations from relying on rare weather categories with insufficient evidence.

By default, the search is restricted to travel windows between 06:00 and 22:00. This is a practical design boundary that prevents very low overnight traffic periods from dominating the recommendations. Users may change these boundaries when calling the function.

This approach is consistent with the earlier association-rule and SHAP findings, which showed that travel timing and recurring calendar patterns were stronger traffic signals than weather alone.

In [5]:
def format_travel_window(hour):
    start_hour = int(hour)
    end_hour = (start_hour + 1) % 24

    return (
        f"{start_hour:02d}:00 to "
        f"{end_hour:02d}:00"
    )


def recommend_travel_windows(
    day_type,
    weather,
    earliest_hour=6,
    latest_hour=22,
    number_of_windows=3,
):
    day_type = str(day_type).strip().title()
    weather = str(weather).strip().title()

    valid_day_types = set(
        data["Day_Type"].dropna().unique()
    )
    valid_weather = set(
        data["Weather"].dropna().unique()
    )

    if day_type not in valid_day_types:
        raise ValueError(
            "day_type must be 'Weekday' or 'Weekend'."
        )

    if weather not in valid_weather:
        raise ValueError(
            "Unknown weather category. Choose from: "
            f"{sorted(valid_weather)}"
        )

    if not 0 <= earliest_hour <= 23:
        raise ValueError(
            "earliest_hour must be between 0 and 23."
        )

    if not 1 <= latest_hour <= 24:
        raise ValueError(
            "latest_hour must be between 1 and 24."
        )

    if earliest_hour >= latest_hour:
        raise ValueError(
            "earliest_hour must be earlier than "
            "latest_hour."
        )

    if number_of_windows < 1:
        raise ValueError(
            "number_of_windows must be at least 1."
        )

    candidates = reliable_weather_patterns[
        (
            reliable_weather_patterns[
                "Day_Type"
            ] == day_type
        )
        & (
            reliable_weather_patterns[
                "Weather"
            ] == weather
        )
        & (
            reliable_weather_patterns[
                "hour"
            ] >= earliest_hour
        )
        & (
            reliable_weather_patterns[
                "hour"
            ] < latest_hour
        )
    ].copy()

    if len(candidates) >= number_of_windows:
        evidence_basis = (
            "Day type and weather-specific "
            "historical pattern"
        )
    else:
        candidates = day_hour_patterns[
            (
                day_hour_patterns[
                    "Day_Type"
                ] == day_type
            )
            & (
                day_hour_patterns[
                    "hour"
                ] >= earliest_hour
            )
            & (
                day_hour_patterns[
                    "hour"
                ] < latest_hour
            )
        ].copy()

        evidence_basis = (
            "Day-type historical pattern; "
            "weather-specific sample was insufficient"
        )

        LOGGER.warning(
            "Insufficient reliable observations for "
            "%s and %s. Using day-type patterns.",
            day_type,
            weather,
        )

    recommendations = (
        candidates.sort_values(
            "Average_Traffic"
        )
        .head(number_of_windows)
        .copy()
    )

    recommendations["Travel_Window"] = (
        recommendations["hour"].apply(
            format_travel_window
        )
    )

    recommendations["Selected_Day_Type"] = day_type
    recommendations["Selected_Weather"] = weather
    recommendations["Evidence_Basis"] = evidence_basis

    recommendations["Recommendation"] = (
        recommendations.apply(
            lambda row: (
                f"Consider travelling between "
                f"{row['Travel_Window']}. "
                f"Historical average traffic was "
                f"approximately "
                f"{row['Average_Traffic']:.0f} "
                f"vehicles per hour."
            ),
            axis=1,
        )
    )

    LOGGER.info(
        "Generated %d travel-window recommendations "
        "for %s conditions on a %s",
        len(recommendations),
        weather,
        day_type,
    )

    return recommendations[
        [
            "Selected_Day_Type",
            "Selected_Weather",
            "Travel_Window",
            "Average_Traffic",
            "Median_Traffic",
            "Observations",
            "Evidence_Basis",
            "Recommendation",
        ]
    ].reset_index(drop=True)

## Scenario testing

The recommendation system is tested using weekday and weekend journeys under different weather conditions. The scenarios include common weather conditions and a rare weather category to demonstrate the system's sample-size fallback.

Each output identifies three comparatively lower-traffic travel windows within practical daytime and evening hours.

In [6]:
test_scenarios = [
    {
        "Scenario": "Weekday travel during rain",
        "day_type": "Weekday",
        "weather": "Rain",
        "earliest_hour": 6,
        "latest_hour": 22,
    },
    {
        "Scenario": "Weekend travel in clear weather",
        "day_type": "Weekend",
        "weather": "Clear",
        "earliest_hour": 6,
        "latest_hour": 22,
    },
    {
        "Scenario": "Weekday travel during squall conditions",
        "day_type": "Weekday",
        "weather": "Squall",
        "earliest_hour": 6,
        "latest_hour": 22,
    },
]

scenario_outputs = []

for scenario in test_scenarios:
    scenario_recommendations = (
        recommend_travel_windows(
            day_type=scenario["day_type"],
            weather=scenario["weather"],
            earliest_hour=scenario[
                "earliest_hour"
            ],
            latest_hour=scenario[
                "latest_hour"
            ],
            number_of_windows=3,
        )
    )

    scenario_recommendations.insert(
        0,
        "Scenario",
        scenario["Scenario"],
    )

    scenario_outputs.append(
        scenario_recommendations
    )

recommendation_results = pd.concat(
    scenario_outputs,
    ignore_index=True,
)

display_columns = [
    "Scenario",
    "Travel_Window",
    "Average_Traffic",
    "Observations",
    "Evidence_Basis",
    "Recommendation",
]

recommendation_results[
    display_columns
].round(
    {
        "Average_Traffic": 0,
    }
)

2026-09-25 04:28:50,120 | INFO | __main__ | Generated 3 travel-window recommendations for Rain conditions on a Weekday
2026-09-25 04:28:50,130 | INFO | __main__ | Generated 3 travel-window recommendations for Clear conditions on a Weekend
2026-09-25 04:28:50,134 | WARNING | __main__ | Insufficient reliable observations for Weekday and Squall. Using day-type patterns.
2026-09-25 04:28:50,138 | INFO | __main__ | Generated 3 travel-window recommendations for Squall conditions on a Weekday


,Scenario,Travel_Window,Average_Traffic,Observations,Evidence_Basis,Recommendation
0,Weekday travel during rain,21:00 to 22:00,2613.0,182,Day type and weather-specific historical pattern,Consider travelling between 21:00 to 22:00. Hi...
1,Weekday travel during rain,20:00 to 21:00,2773.0,168,Day type and weather-specific historical pattern,Consider travelling between 20:00 to 21:00. Hi...
2,Weekday travel during rain,19:00 to 20:00,3256.0,188,Day type and weather-specific historical pattern,Consider travelling between 19:00 to 20:00. Hi...
3,Weekend travel in clear weather,06:00 to 07:00,1117.0,171,Day type and weather-specific historical pattern,Consider travelling between 06:00 to 07:00. Hi...
4,Weekend travel in clear weather,07:00 to 08:00,1632.0,147,Day type and weather-specific historical pattern,Consider travelling between 07:00 to 08:00. Hi...
5,Weekend travel in clear weather,08:00 to 09:00,2414.0,166,Day type and weather-specific historical pattern,Consider travelling between 08:00 to 09:00. Hi...
6,Weekday travel during squall conditions,21:00 to 22:00,2673.0,1425,Day-type historical pattern; weather-specific ...,Consider travelling between 21:00 to 22:00. Hi...
7,Weekday travel during squall conditions,20:00 to 21:00,2842.0,1418,Day-type historical pattern; weather-specific ...,Consider travelling between 20:00 to 21:00. Hi...
8,Weekday travel during squall conditions,19:00 to 20:00,3297.0,1407,Day-type historical pattern; weather-specific ...,Consider travelling between 19:00 to 20:00. Hi...


## Saving recommendation outputs

The tested travel-window recommendations are saved as a CSV file. This provides a reproducible output that can be reviewed, included in reporting and connected to the deployment interface.

In [7]:
results_path = (
    RECOMMENDATION_DIR
    / "travel_timing_recommendations.csv"
)

saved_recommendations = (
    recommendation_results.copy()
)

saved_recommendations[
    "Average_Traffic"
] = saved_recommendations[
    "Average_Traffic"
].round(1)

saved_recommendations[
    "Median_Traffic"
] = saved_recommendations[
    "Median_Traffic"
].round(1)

saved_recommendations.to_csv(
    results_path,
    index=False,
)

LOGGER.info(
    "Saved %d travel-window recommendations to %s",
    len(saved_recommendations),
    results_path,
)

print(
    "Saved recommendation results:",
    results_path,
)

2026-09-25 04:29:38,411 | INFO | __main__ | Saved 9 travel-window recommendations to /Users/misatoniwa/Downloads/smart-city-traffic-capstone/part3_machine_learning/recommendation_system/travel_timing_recommendations.csv
Saved recommendation results: /Users/misatoniwa/Downloads/smart-city-traffic-capstone/part3_machine_learning/recommendation_system/travel_timing_recommendations.csv


## Interpretation and limitations

The recommendation system identifies practical lower-traffic travel windows using historical hourly traffic patterns. For the tested scenarios, weekday travel during rain was generally more suitable later in the evening, while weekend travel in clear weather was more suitable during the early morning. These findings are consistent with the earlier association-rule and SHAP analyses, which showed that time-based and calendar-based patterns were stronger traffic signals than weather alone.

The system also accounts for the reliability of its evidence. Weather-specific recommendations are only produced when at least 30 historical observations are available. When this requirement is not met, as demonstrated for Squall conditions, the system uses the broader day-type pattern and clearly reports the fallback.

The minimum requirement of 30 observations is a manually selected reliability threshold rather than an operationally validated standard. Different thresholds could change which recommendations use weather-specific evidence.

The recommendations are limited to one monitored road corridor and are based on historical averages rather than live traffic predictions. They do not account for incidents, roadworks, special events or sudden changes in weather. Rare weather categories may also have insufficient observations for weather-specific recommendations. The outputs should therefore support travel planning rather than guarantee future traffic conditions.